# **Spheroids in microcapsules**
Authors: Kazuki Hattori, Yuichiro Iwamoto

In [ ]:
!pip install imagecodecs
!pip install pyclesperanto_prototype

## Connect with Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Import libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator
from scipy import io, ndimage as ndi, signal
from skimage import io as skio, color, filters, segmentation, feature, measure, morphology, transform, draw
from skimage.filters import median as skimage_filters_median, threshold_otsu, threshold_yen, threshold_multiotsu, gaussian
from skimage.io import imread
from skimage.measure import regionprops, regionprops_table, label
from skimage.morphology import erosion, dilation, square, disk
from skimage.segmentation import clear_border, watershed
from skimage.feature import peak_local_max
from skimage.draw import circle_perimeter
from skimage.color import label2rgb
import random
from datetime import datetime, timedelta, timezone
import pytz
import os
import seaborn as sns
import math
import time
import glob
from PIL import Image
from tqdm import tqdm
import gc
import pyclesperanto_prototype as cle

##Preset

In [ ]:
CodeName = "Spheroid_in_microcapsule"
Version = "1.19.20240915"
Python_version = !python --version
exp_id = "XXX"
sample_id ="XXX"
main_path = "XXX"
save_path_fig = "XXX"
save_path_data_sheet = "XXX"

##Functions

In [ ]:
#select_circular_labels
def select_circular_labels(binary_img, circularity_threshold):
    img_labeled = label(binary_img)
    props_df = pd.DataFrame(regionprops_table(img_labeled, properties=('label', 'area', 'perimeter', 'eccentricity', 'solidity')))
    props_df['circularity'] = (4 * np.pi * props_df['area']) / (props_df['perimeter'] ** 2)
    circular_labels = props_df[props_df['circularity'] > circularity_threshold]
    img_circular = np.isin(img_labeled, circular_labels['label'])

    return img_circular

#binarization
def binarization(img, gaussian_sigma, threshold_method, watershed_apply, binarization_apply, signal_threshold, circularity_threshold, threshold_variance_triangle_apply):
  if binarization_apply == 1:
    if threshold_variance_triangle_apply == 1:
      img = threshold_variance_triangle(img)
    else:
      #denoise
      img = skimage_filters_median(img)
      img = gaussian(img, sigma = gaussian_sigma)
      #binarization
      img_thresh = threshold_method(img)
      img = img >= img_thresh
      #erosion and dilation
      img = erosion(img, square(ero))
      img = dilation(img, square(dil))
    #fill holes
    img = ndi.binary_fill_holes(img)
    #clear border
    img = clear_border(img)
    #watershed
    if watershed_apply == 1 :
      distance = ndi.distance_transform_edt(img)
      coords = peak_local_max(distance, footprint=np.ones(watershed_footprint), labels=img)
      local_maxi = peak_local_max(distance, indices=False, footprint=np.ones(watershed_footprint), labels=img)
      markers, _ = ndi.label(local_maxi)
      img  = watershed(-distance, markers, mask=img)
    else:
      pass
    select_circular_labels(img, circularity_threshold)

  else:
    img = img >= signal_threshold

  return img

#calculate ratio
def ratio(df, pairs):
    for small, large in pairs:
        df[f"{small}_to_{large}"] = df[small] / df[large]
    return df

#rename the name of the columns
def rename_df_columns(df, label_list_name, label_num, channel_name = None, channel_num = None):
    if channel_name is not None and channel_num is not None:
      suffix = f"{label_list_name[label_num]}-mask_{channel_name[channel_num]}"
    else:
      suffix = f"{label_list_name[label_num]}-mask"
    new_columns = {
        'intensity_mean': f"{suffix}_mean_intensity",
        "centroid-0": f"{suffix}_centroid-0",
        "centroid-1": f"{suffix}_centroid-1",
        "area": f"{suffix}_area"
    }
    df.rename(columns=new_columns, inplace=True)
    return df

#set time
utc_now = datetime.utcnow().replace(tzinfo=pytz.utc)
jst_now = utc_now.astimezone(pytz.timezone("Asia/Tokyo"))
timestamp_short = jst_now.strftime("%Y%m%d_%H%M")

#calculate diameters of approximate circles
def calculate_diameter(label):
    blank_image_circle = np.zeros_like(label)
    diameters = []

    for region in regionprops(label):
        minr, minc, maxr, maxc = region.bbox
        height = maxr - minr
        width = maxc - minc
        diameter = 2 * max(height, width) / 2
        diameters.append(diameter)

        # Calculate center and draw circle perimeter
        center_y, center_x = (minr + height // 2, minc + width // 2)
        radius = int(max(height, width) / 2)

        # Check if the circle will fit within the image
        if (center_x - radius >= 0 and center_x + radius < blank_image_circle.shape[1] and
            center_y - radius >= 0 and center_y + radius < blank_image_circle.shape[0]):
            rr, cc = circle_perimeter(center_y, center_x, radius)
            blank_image_circle[rr, cc] = 255  # Draw circle
        else:
            continue

    return blank_image_circle

#collect intensities from many images - for determining threshold values
def collect_pixel_intensities(main_path, channel_id, loop):
    tif_files_d = sorted([f for f in os.listdir(main_path) if f.lower().endswith("d" + str(channel_id) + '.tif')])
    pixel_values = []
    for i in range(loop):
        img = imread(os.path.join(main_path, tif_files_d[i]), plugin='pil')
        img_array = np.array(img).flatten()
        pixel_values.extend(img_array)
    pixel_values = np.array(pixel_values)
    pixel_values = pixel_values.reshape(1, -1)

    return pixel_values[0]

#change label colors
def change_label_color(imgs, label_list_figure, color, channel_number):
  imgs_label = np.zeros((channel_num, num_steps, imgs.shape[2], imgs.shape[3], 3), dtype=np.int64)
  for k in label_list_figure:
    for j in range(channel_number):
      if color == "None":
         imgs_label[j, k] = label2rgb(imgs[j, k], bg_label=0, bg_color=None)
      else:
        imgs_label[j, k] = label2rgb(imgs[j, k], bg_label=0, colors=[color], bg_color=None)
  return imgs_label

 #annotate labels on figures
def annotate_axes_with_labels(df, ax_array, cent_y, cent_x, font_size):
    num_rows, num_cols = ax_array.shape
    for j, label in enumerate(df['label']):
        centroid_x = df[cent_x][j] + 50  # Adjusted x coordinate
        centroid_y = df[cent_y][j] + 50  # Adjusted y coordinate
        text = label  # The label to annotate

        # Loop through each subplot in the ax_array and add text annotation
        for row in range(num_rows):
            for col in range(num_cols):
                ax_array[row][col].text(x=centroid_x, y=centroid_y, s=text, size=font_size, color='white')

#skip image if vacant
def get_img(imgs, idx, sub_idx, default_value=np.zeros((1536, 2048))):
    try:
        return imgs[idx][sub_idx]
    except IndexError:
        return default_value

def threshold_variance_triangle (img):
    footprint = disk(1)
    elevation_map =  ndi.gaussian_filter(np.array(cle.variance_box(img, radius_x=1, radius_y=1)),sigma=1)
    th=filters.threshold_triangle(elevation_map)
    img_=ndi.binary_opening(elevation_map>th,structure=np.ones((5,5)))
    img_=ndi.binary_fill_holes(ndi.binary_dilation(img_,structure=np.ones((4,4))))

    return img_

##Pre-analysis for determining threshold values of SYTOX

In [ ]:
#preset
main_paths = ["XXX"]
channel_id_list = [3]
channel_name = ["LipiBlue", "FITC-agarose", "CytoRed","SYTOX", "PC"]
sampling_ratio = 0.01
timestamp_short = datetime.utcnow().replace(tzinfo=pytz.utc).astimezone(pytz.timezone("Asia/Tokyo")).strftime("%Y%m%d_%H%M")#set time

file_names_used = []
for i in range(len(channel_id_list)):
  combined_d0_list = []

  for main_path in main_paths:
    files_in_folder = sorted([f for f in os.listdir(main_path) if f.lower().endswith("d" + str(channel_id_list[i]) + '.tif')])
    file_names_used.extend([os.path.join(main_path, f) for f in files_in_folder])
    field_num = len(files_in_folder)
    d0_list = collect_pixel_intensities(main_path, channel_id_list[i], field_num)
    combined_d0_list.extend(d0_list)

  bin_width = 20
  tick_interval = 200

  combined_d0_array = np.array(combined_d0_list)
  thresholds = threshold_multiotsu(combined_d0_array, classes=3)

  sampled_data = np.random.choice(combined_d0_array, size = int(len(combined_d0_array) * sampling_ratio), replace=False)

  #histogram
  sns.set_style("whitegrid")
  sns.set_context("notebook")
  fig, ax = plt.subplots()
  hist_data = sns.histplot(sampled_data, kde=False, color="skyblue", edgecolor='none', binwidth=bin_width, ax=ax)
  #sns.histplot(d0_list2, kde=False, color="orange", edgecolor='none', binwidth=bin_width, ax=ax, alpha = 0.3)
  heights = [rect.get_height() for rect in hist_data.patches]
  y_max = 0.001 * max(heights) if heights else 0
  plt.axvline(x = thresholds[0], color='blue', linestyle='--', label=f"Multi-Otsu's_threshold_1st: {thresholds[0]}", linewidth=0.5)
  plt.axvline(x = thresholds[1], color='magenta', linestyle='--', label=f"Multi-Otsu's_threshold_2nd: {thresholds[1]}", linewidth=0.5)
  ax.xaxis.set_major_locator(MultipleLocator(tick_interval))
  plt.legend()
  plt.xticks(fontsize=8, rotation = 45)
  ax.set(xlabel="Signal intensities of " + channel_name[channel_id_list[i]])
  ax.set_ylim(None, y_max)
  plt.annotate("pixel_number_analyzed = " + str(len(combined_d0_list)) + ", pixel_number_used_for_histogram = " + str(len(sampled_data)), (0,1.1), xycoords='axes fraction')
  ax.figure.savefig("XXX" + ".png", dpi=300, transparent=True, bbox_inches="tight")
  plt.show()


#save summary
summary_lines = [
    "",
    f"Code: {CodeName}_Ver. {Version}",
    f"Python_version: {Python_version}",
    f"Date: {datetime.now(pytz.timezone('Asia/Tokyo')).strftime('%Y-%m-%d %H:%M:%S')}",

    "",
    "List of TIFF Files:"
]
summary_lines.extend(file_names_used)
summary_lines.append("")

with open("XXX"+ "_Summary.txt", 'a') as f:
    for line in summary_lines:
        f.write(line + '\n')


##Analysis

In [ ]:
for s in [exp_list]:
  exp_id = str(s)
  main_path = "XXX"

  image_export = 1 #export images if 1
  only_one_image = 0 #analyze only the 1st field
  only_one_image_id = 1 # image id list for the analysis when only_one_image = 1
  export_in_svg = 0 #export in svg not in PNG

  tif_files = sorted([f for f in os.listdir(main_path) if f.lower().endswith('.tif')])

  channel_num = 5 #per field
  num_steps = 13 #total number of raw and processed images
  imgs = np.zeros((channel_num, num_steps, *imread(os.path.join(main_path, tif_files[0]), plugin='pil').shape), dtype=np.int64)

  channel_name = ["LipiBlue", "FITC-agarose", "CytoRed", "SYTOX", "PC"] #d0 to d4
  skip_flag_diameter = [0,1,0,0,0] #calculate approximate circle if 1, skip if 0
  gaussian_sigma_list = [1,1,1,1,1]
  ero = 5 #kernel for erosion, square
  dil = 5 #kernel for dilation, square
  binarization_list = [1, 1, 1, 0, 1] #binarization if 1, slice data with the values in signal_threshold_list if 0
  threshold_list = [threshold_otsu, threshold_otsu, threshold_otsu, 0, 0] #binarization methods where binarization_list = 1
  threshold_variance_triangle_apply_list = [0, 0, 0, 0, 1]
  signal_threshold_list =[0,0,0,1159,0] #lower limit of the signal intensity - set no threshold if 0 - usually set these values by Otsu but manually sometimes
  watershed_list = [0, 0, 0, 0, 0] #perform watershed if 1
  watershed_footprint = (3,3)
  circularity_threshold_list = [0, 0.8, 0, 0, 0]
  circularity_threshold_list_after_merge = [0.2, 0, 0, 0, 0] #for imgs[,11]
  area_threshold_list = [819, 15000, 819, 0, 819] #lower area limit of the label - no threshold if 0 #819, 1842 = area of circle 20 or 30 um in diameter
  upper_area_threshold_list = [100000, 50000, 100000, 100000, 100000] #upper area limit of the label
  area_threshold_list_circle = [0, 500, 0, 0, 0] #lower area limit of the circle label - no threshold if 0
  upper_area_threshold_list_circle = [100000, 100000, 100000, 100000, 100000] #upper area limit of the circle label
  shell_label_id = 2 #binarized label is 2 and approximate circle is 6
  shell_label_channel_id = 1
  ratio_list = [ ('LipiBlue-in-spheroid-in-shell-mask_area', 'Spheroid-in-shell-mask_area'), ('SYTOX-in-spheroid-in-shell-mask_area', 'Spheroid-in-shell-mask_area')] #use channel names defined above
  mask_list = [imgs[4,1], imgs[0,1], imgs[3,1], imgs[2,1]] #for overlaying masks
  label_list = [imgs[1,shell_label_id], imgs[1,7], imgs[1,8], imgs[1,9], imgs[1,10]] # for regionprops
  label_list_name = ["Shell","Spheroid-in-shell", "LipiBlue-in-spheroid-in-shell", "SYTOX-in-spheroid-in-shell", "Spheroid-in-shell_CytoRed"]
  regionprops_tuple_initial = ("label", "centroid", "area")
  regionprops_tuple = ("label", "intensity_mean",)
  merge_channel_id = 0 #merege binarized data to prepare additional label file - imgs[j,11] is binary and imgs[j,12] is label

  summary_lines = [
      "",
      f"Code: {CodeName}_Ver. {Version}",
      f"Python_version: {Python_version}",
      f"Date: {datetime.now(pytz.timezone('Asia/Tokyo')).strftime('%Y-%m-%d %H:%M:%S')}",
      f"Experiment ID: {exp_id}",
      f"Sample ID: {sample_id}",
      f"File Path: {main_path}",
      f"Number of TIFF Files: {len(tif_files)}",
      f"Number of Channels per Field: {channel_num}",
      f"Total Steps (Raw & Processed Images): {num_steps}",
      f"Channel Names: {', '.join(channel_name)}",
      f"Skip Flags for Diameter Calculation: {skip_flag_diameter}",
      f"Gaussian Sigma Values: {gaussian_sigma_list}",
      f"Erosion Kernel Size: {ero}",
      f"Dilation Kernel Size: {dil}",
      f"Watershed Application per Channel: {watershed_list}",
      f"Lower Area Thresholds per Channel: {area_threshold_list}",
      f"Upper Area Thresholds per Channel: {upper_area_threshold_list}",
      f"Lower Circle Area Thresholds per Channel: {area_threshold_list_circle}",
      f"Upper Circle Area Thresholds per Channel: {upper_area_threshold_list_circle}",
      f"Circularity Threshold: {circularity_threshold_list}",
      f"Circularity Threshold After Merging : {circularity_threshold_list_after_merge}",
      f"Binarization Settings per Channel: {binarization_list}",
      f"Binarization Methods: {threshold_list}",
      f"Binarization by threshold_variance_triangle: {threshold_variance_triangle_apply_list}",
      f"Mask list: {mask_list}",
      f"Label list: {label_list}",
      f"Label list name: {label_list_name}",
      f"Signal Thresholds per Channel: {signal_threshold_list}",
      f"Area Ratios Used for Analysis: {', '.join([f'{x[0]} / {x[1]}' for x in ratio_list])}",
      f"Labels for Region Properties: {', '.join(label_list_name)}",
      f"Initial Properties for Region Analysis: {', '.join(regionprops_tuple_initial)}",
      f"Extended Properties for Region Analysis: {', '.join(regionprops_tuple)}",
      "",
      "List of TIFF Files:"
  ]


  timestamp_short = datetime.utcnow().replace(tzinfo=pytz.utc).astimezone(pytz.timezone("Asia/Tokyo")).strftime("%Y%m%d_%H%M")#set time

  #analysis
  df_all = pd.DataFrame()

  if only_one_image == 1:
    for_list_analysis = only_one_image_id
  else:
    for_list_analysis = range(int(len(tif_files)/channel_num))

  for n in for_list_analysis:
    for j in range(channel_num):
      try:
          imgs[j, 0] = imread(os.path.join(main_path, tif_files[j+channel_num*n]), plugin='pil')
      except FileNotFoundError:
          imgs[j, 0] = np.zeros_like(imread(os.path.join(main_path, tif_files[0]), plugin='pil'))
      imgs[j, 1] = binarization (imgs[j,0],gaussian_sigma_list[j],threshold_list[j], watershed_list[j], binarization_list[j], signal_threshold_list[j], circularity_threshold_list[j],threshold_variance_triangle_apply_list[j]) #boolean mask
      labeled_img, _ = ndi.label(imgs[j, 1])
      imgs[j, 1] = np.isin(labeled_img, [prop.label for prop in regionprops(labeled_img) if area_threshold_list[j] <= prop.area <= upper_area_threshold_list[j]]) #select labels based on the area and transfer back to boolean mask
      imgs[j, 2], _ = ndi.label(imgs[j,1])
      if skip_flag_diameter[j]: #transform the labels to circle labels
        imgs[j, 3] = calculate_diameter(imgs[j,2]) #modify labels in [j,3] to the circles, not filled
        imgs[j, 4] = ndi.binary_fill_holes(imgs[j, 3]) #fill holes of the circles in [j,3]
        imgs[j, 5], _ = ndi.label(imgs[j, 4]) #label number
        imgs[j, 6] = np.where((np.bincount(imgs[j, 5].ravel())[imgs[j, 5]] >= area_threshold_list_circle[j]) & (np.bincount(imgs[j, 5].ravel())[imgs[j, 5]] <= upper_area_threshold_list_circle[j]), imgs[j, 5], 0) #select labels based on the area
      else:
        imgs[j, 3:7] = imgs[j, 2]

    for j in range(channel_num):
      imgs[j, 11] = imgs[j, 1]+ imgs[merge_channel_id,1]
      imgs[j, 11] = (imgs[j, 11] > 0).astype(int)
      imgs[j, 11] = select_circular_labels(imgs[j,11], circularity_threshold_list_after_merge[j])
      imgs[j, 12], _ = ndi.label(imgs[j,11])

    for j in range(channel_num):
      imgs[j, 7] = imgs[j, shell_label_id]*mask_list[0] #additional label
      imgs[j, 8] = imgs[j, shell_label_id]*mask_list[0]*mask_list[1] #additional label
      imgs[j, 9] = imgs[j, shell_label_id]*mask_list[0]*mask_list[2] #additional label
      imgs[j, 10] = imgs[j, shell_label_id]*mask_list[3] #additional label


    for k in range(len(label_list)):
      df_channel = pd.DataFrame(regionprops_table(label_list[k], imgs[0,0], properties=regionprops_tuple_initial))
      rename_df_columns(df_channel, label_list_name, k)
      for m in range(channel_num):
        df_channel_mask = pd.DataFrame()
        df_channel_mask = pd.DataFrame(regionprops_table(label_list[k], imgs[m,0], properties=regionprops_tuple))
        rename_df_columns(df_channel_mask, label_list_name, k, channel_name, m)
        df_channel = pd.merge(df_channel, df_channel_mask, how="outer", on="label")
      if k == 0:
        df = df_channel
      else:
        df = pd.merge(df, df_channel, how='outer', on='label')

    df = ratio(df, ratio_list)
    df.insert(loc = 0, column= "image_id", value= n)
    df.insert(loc = 0, column= "sample_id", value= sample_id)
    df.insert(loc = 0, column= "exp_id", value= exp_id)

    df_all = pd.concat([df_all, df]).fillna(0)

  #Export images
    if image_export == 1:
      alpha_list = [1, 0.2]
      label_colors = [None] * 13
      label_colors[0] = label2rgb(get_img(imgs, 0, 1), bg_label=0, colors=["blue"])
      label_colors[1] = label2rgb(get_img(imgs, 1, 1), bg_label=0, colors=["green"])
      label_colors[2] = label2rgb(get_img(imgs, 2, 1), bg_label=0, colors=["orange"])
      label_colors[3] = label2rgb(get_img(imgs, 3, 1), bg_label=0, colors=["red"])
      label_colors[4] = label2rgb(get_img(imgs, 4, 1), bg_label=0, colors=["pink"])
      label_colors[5] = label2rgb(get_img(imgs, 1, 7), bg_label=0, colors=["white"])
      label_colors[6] = label2rgb(get_img(imgs, 1, 8), bg_label=0, colors=["cyan"])
      label_colors[7] = label2rgb(get_img(imgs, 1, 9), bg_label=0, colors=["magenta"])
      label_colors[8] = label2rgb(get_img(imgs, 0, 12), bg_label=0, colors=["blue"])
      label_colors[9] = label2rgb(get_img(imgs, 1, 12), bg_label=0, colors=["green"])
      label_colors[10] = label2rgb(get_img(imgs, 2, 12), bg_label=0, colors=["orange"])
      label_colors[11] = label2rgb(get_img(imgs, 3, 12), bg_label=0, colors=["red"])
      label_colors[12] = label2rgb(get_img(imgs, 4, 12), bg_label=0, colors=["pink"])
      shell_label = label2rgb(imgs[shell_label_channel_id,6], bg_label=0, colors=["yellow"])

      images_and_titles = []
      for i in range(channel_num):
          images_and_titles.append(([imgs[i][0]], channel_name[i], None, "gray"))
      for i in range(channel_num):
          images_and_titles.append(([label_colors[i]], channel_name[i] + "_label", None, None))
      for i in range(channel_num):
          images_and_titles.append(([imgs[channel_num-1][0], label_colors[i]], channel_name[channel_num-1] + " + " + channel_name[i] + "_label", alpha_list, "gray"))
      for i in range(channel_num):
          images_and_titles.append(([imgs[i][0], label_colors[i]], channel_name[i] + " + " + channel_name[i] + "_label", alpha_list, "gray"))
      for i in range(8,8+channel_num):
          images_and_titles.append(([label_colors[i]], "[" + channel_name[i-8] + " + " + channel_name[merge_channel_id] + "]_label", None, None))
      if channel_num == 4:
          for i in range(8, 8 + channel_num):
              images_and_titles.append(([imgs[3][0], label_colors[i]], "PC+[" + channel_name[i-8] + " + " + channel_name[merge_channel_id] + "]_label", alpha_list, "gray"))
      else:
          pass
      if channel_num == 5:
          for i in range(8, 8 + channel_num):
              images_and_titles.append(([imgs[4][0], label_colors[i]], "PC+[" + channel_name[i-8] + " + " + channel_name[merge_channel_id] + "]_label", alpha_list, "gray"))
      else:
          pass
      if channel_num == 2:
        images_and_titles.extend([
            ([imgs[1][0], shell_label], channel_name[shell_label_channel_id] + " + " + channel_name[1] + "_label-filtered_circle", alpha_list, "gray"),
            ([shell_label], channel_name[shell_label_channel_id] + "_label-filtered_circle", None, None),
        ])
      else:
        pass

      fig, ax = plt.subplots(6, channel_num, figsize=(20, 16))
      for idx, (imgs_fig, title, alphas, color) in enumerate(images_and_titles):
          row, col = divmod(idx, channel_num)
          for i, img_to_plot in enumerate(imgs_fig):
              alpha = 1 if alphas is None else alphas[i]
              ax[row][col].imshow(img_to_plot, alpha=alpha, cmap = color)
          ax[row][col].axis('off')
          ax[row][col].set_title(title)

      if channel_num == 5:
        for j, l in enumerate(df['label']):
          for i in range(5):
              ax[3][i].text(x=df['Spheroid-in-shell-mask_centroid-1'][j], y=df['Spheroid-in-shell-mask_centroid-0'][j], s=l, size=9, color='white')
      else:
        pass

      if channel_num == 4:
          for j, l in enumerate(df['label']):
              for i in range(4):
                  ax[4][i].text(x=df['Spheroid-mask_centroid-1'][j], y=df['Spheroid-mask_centroid-0'][j], s=l, size=9, color='white')
      else:
          pass

      plt.tight_layout()
      plt.annotate(exp_id + "_" + sample_id + "_image_id_"+ str(n) + "_Otsu's binarization per field: " + str([channel_name[m] for m in range(channel_num) if binarization_list[m] == 1]) , (-1,7.1), xycoords='axes fraction')
      #plt.show()
      plt.rcParams["svg.fonttype"] = "none"
      if export_in_svg == 1:
        fig.savefig("XXX" + ".svg", format = 'svg', transparent=True, bbox_inches="tight")
      else:
        fig.savefig("XXX" + ".jpg", dpi = 300, bbox_inches="tight")
      plt.close("all")
    else:
      pass

  df_all.to_csv("XXX" +  ".csv", index=False)

  #save summary
  summary_lines.extend(tif_files)
  summary_lines.append("")

  with open("XXX" + "_Summary.txt", 'a') as f:
      for line in summary_lines:
          f.write(line + '\n')
